# Modelagem preditiva e avaliação de classificadores (REVISÃO_SOMATIVA01.ipynb)

- **Repositório:** `machine-learning-portfolio`  
- **Tecnologias:** Scikit-Learn  
- **Arquivo Original:** `REVISÃO_SOMATIVA01.ipynb`  

---


# Revisão Somativa 01

### Importar bibliotecas

In [1]:
import numpy as np
import scipy.stats as st

## Conectar com o Google Drive

In [18]:
from google.colab import drive
drive.mount('/content/gdrive')

MessageError: Error: credential propagation was unsuccessful

## Variáveis de ambiente para arquivos e conversão para PDF

In [ ]:
# Path para pasta de DADOS
caminho_dados = '/content/gdrive/MyDrive/DADOS/ADS/'

# Path para pasta de PDF
caminho_pdf = '/content/gdrive/MyDrive/PDF/ADS'

# Nome do notebook
nome = 'REVISÃO_SOMATIVA01'

## Processos de Poisson

### Questão 1
Um grupo de mini-VANs está esperando passageiros no aeroporto. O tempo entre duas chegadas consecutivas de passageiros é distribuído exponencialmente com média de 3 passageiros por minuto. Uma mini-VAN parte assim que tem 5 passageiros ou assim que se passe 10 minutos desde que o primeiro passageiro entrou na mini-VAN. Suponha que você foi o primeiro passageiro a entrar na min-VAN. Qual é a probabilidade que você tenha que esperar 10 minutos até a saída da mini-VAN?<br><br>
Dicas: Você pode calcular a probabilidade de duas maneiras:<br>
(i) Para que você espere 10 minutos, a chegada do quarto passageiro depois de você deve ocorrer após 10 minutos. Isso corresponde à soma de 4 variáveis aleatórias exponenciais com média 3 ($mu=3)$ ser maior do que 10. Usar a equação da função gama para calcular $P[S_{N,mu} > x]$.<br>
$P\left[ S_{N,MU} = x \right] =$ 1- st.gamma.cdf(x, a=N, scale=mu)<br><br>
(ii) Para que você espere mais do que 10 minutos é preciso que no intervalo de 10 minutos ocorra uma das seguintes possibilidades: nenhum passageiro chegou no intervalo de 10 minutos, ou 1 ou 2 ou 3 passageiros chegaram. O quarto passageiro deve chegar depois de 10 minutos. Usar a equação do processo de poisson para calcular $P[N_{s} = 0] + P[N_{s} = 1] + P[N_{S} = 2] + P[N_{S} = 3]$.<br>
$P[N_{S,L} = x]= $ st.poisson.pmf(x, L*S) com $L =1/MU$.

Escolher uma delas para responder.

In [2]:
# P[SN>x]

x = 10
N = 4
mu = 3


resposta = 1- st.gamma.cdf(x, a=N, scale=mu)
print(resposta)

0.5729859919108071


In [ ]:
# P[Ns=0] + P[Ns=1] + P[Ns=2] + P[Ns=3]



### Questão 2
Um servidor de aplicações trata apenas duas categorias de transações: leitura e atualização. As transações chegam segundo processos de Poisson com as seguintes taxas: $L_{L}$ = 3 transação de leitura por milissegundo e $L_{A}$ = 1 transações de atualização por milissegundo. Qual a probabilidade que em 2 milissegundos cheguem 4 transações, todas de leitura?
Dica: para que cheguem 4 transações, todas de leitura, é necessário que cheguem 4 transações de leitura **E** 0 transações atualização.<br>
$P[N_{S} = x]= $ st.poisson.pmf(x, L*S).

In [10]:
# 4 transações de leitura em 2 milissegundos
LL = 3
S = 2
x = 4

resposta2 = st.poisson.pmf(x, LL*S)
print(resposta2)


# Zero transações de atualização em 200 milissegundos

LA = 1
S = 200
x = 0

resposta3 = st.poisson.pmf(x, LA*S)
print(resposta3)


# 3 transações, todas de atualização, em 200 milissegundos
# Lembrar que a probabilidade da ocorrência de dois eventos (P[evento1 E evento2] = P[evento1] * P[evento2] )

S = 200
x = 3

resposta4 = (st.poisson.pmf(x, LL*S) * st.poisson.pmf(x, LA*S))
print(resposta4)


0.13385261753998332
1.3838965267367376e-87
0.0


## Fila MMm

In [12]:
import numpy as np
from math import factorial
from math import exp

class FilaMMm:
    def calculaP0(self):
        soma = 0
        for n in range(1, self.m):
            soma = soma + (((self.m * self.ro) ** n) / factorial(n))
        return 1 / (1 + ((self.m * self.ro) ** self.m) / (factorial(self.m) * (1 - self.ro)) + soma)

    def __init__(self, lb, mu, m):
        if (lb >= m*mu):
            raise ValueError('Lambda deve ser menor do que m*mu')
        self.lb = float(lb)
        self.mu = float(mu)
        self.m = m
        self.ro = lb / (m*mu)
        self.p0 = self.calculaP0()
        self.epsilon = ((self.m*self.ro)**self.m)/((1-self.ro)*factorial(self.m))*self.p0
        # valor esperado de Ns (quantidade media de tarefas servidas)
        self.E_Ns = self.m * self.ro
        # valor esperado de Nq (tamanho medio da fila)
        self.E_Nq = (self.epsilon * self.ro) / (1 - self.ro)
        # valor esperado de N 9quentidade media de tarefas no sistema)
        self.E_N = self.E_Nq + self.E_Ns
        # valor esperado de S (tempo de serviço médio)
        self.E_S = 1 / self.mu
        # valor esperado de W (tempo de espera medio na fila)
        self.E_W = self.epsilon / (self.m * self.mu * (1 - self.ro))
        # valor esperado de R (tempo de resposta medio)
        self.E_R = self.E_S + self.E_W


    def pmf_N(self, x):
        if (x<0 or not(isinstance(False, int))):
            raise ValueError('x deve ser inteiro positivo')
        if (x < self.m):
            return (((self.m * self.ro) ** x) / factorial(x)) * self.p0
        else:
            return (((self.ro ** x) * (self.m ** self.m)) / factorial(self.m)) * self.p0

    def pmf_Ns(self, x):
        if (x<0 or not(isinstance(False, int))):
            raise ValueError('x deve ser inteiro positivo')
        if (x < self.m):
            return self.pmf_N(x)
        if (x == self.m):
            return self.epsilon
        else:
            return 0.0

    def pmf_Nq(self, x):
        if (x<0 or not(isinstance(False, int))):
            raise ValueError('x deve ser inteiro positivo')
        if (x == 0):
            pNq = 0
            for i in range(0, self.m +1):
                pNq = pNq + self.pmf_N(i)
            return pNq
        else:
            return self.pmf_N(x+self.m)

    def cdf_W(self, x):
        if (x<0):
            raise ValueError('x deve ser positivo')
        return 1 - (self.epsilon*exp(-self.m*self.mu*(1-self.ro)*x))

    def cdf_R(self, x):
        if (x<0):
            raise ValueError('x deve ser positivo')
        if (self.ro != (self.m-1)/self.m):
            MULT = (self.epsilon / (1 - self.m + self.m * self.ro)) # calculado aqui para legibilidade
            p = 1 - exp(-self.mu * x) - MULT * (exp(-self.m * self.mu * (1 - self.ro) * x) - exp(-self.mu * x))
        else:
            p = 1 - exp(-self.mu * x) - (
                    self.epsilon * self.mu * x * exp(-self.mu * x))
        return p

### Questão 3
Um sistema de armazenagem consiste em 4 data storages que compartilham uma fila comum. A taxa de serviço em cada data storage (servidor) é de 20 solicitações por mili-segundos. As solicitações chegam ao sistema a uma taxa de 60 solicitações por mili-segundo.<br>
a)  Criar a fila.<br>
b)	Calcular o número médio de tarefas no sistema.<br>
c)	Calcular a probabilidade de o tempo na fila ser maior do que 0,02 mili-segundos.

In [19]:
# Criar fila
fila = FilaMMm(lb=60, mu=20, m=4)


# Número médio de tarefas no sistema
# Dica: a variável N conta o número de tarefas no sistema
# O valor deve ser 4.528301886792453
fila1 = fila.E_N
print(fila1)



# Probabilidade de o tempo na fila ser maior do que 0,02 milisegundos
# Dicas
# A variável W conta o tempo na fila
# O médoto CDF_W calcula P[W<=x]
# para calcular P[W>x] usar 1-cdf_W
# O valor deve ser 0.3414837970370238
resposta5 = 1 - fila.cdf_W(0.02)
print(resposta5)


4.528301886792453
0.3414837970370238


## Fila MG1

In [21]:
from math import sqrt
class FilaMG1:
    def __init__(self, lb, E_S, DP_S):
        if (lb >= 1/E_S):
            raise ValueError('Lambda deve ser menor do que 1/E_S')
        self.lb = float(lb)
        self.E_S = float(E_S)
        self.DP_S = DP_S
        self.ro = lb * E_S
        self.CV_S = DP_S / E_S

        # valor esperado de Ns (tamanho medio da fila)
        self.E_Ns = self.ro
        # valor esperado de Nq (tamanho medio da fila)
        self.E_Nq = (self.ro**2)*(1 +  self.CV_S**2) / (2*(1 - self.ro))
        # valor esperado de N quantidade media de tarefas no sistema)
        self.E_N = self.E_Nq + self.E_Ns
        # valor esperado de W (tempo médio de espera na fila)
        self.E_W = (self.ro*self.E_S) * (1 +  self.CV_S**2) / (2*(1 - self.ro))
        # valor esperado de R (tempo de resposta medio)
        self.E_R = self.E_S + self.E_W

### Questão 4
Um servidor web recebe 25 solicitações por mili-segundo, sendo exponencial a distribuição do tempo entre duas chegadas consecutivas. A distribuição do tempo de serviço é desconhecida, mas a média é 0,034 mili-segundos, e o desvio padrão é 0,071 mili-segundos. Calcular:
* O número médio de tarefas na fila.
* O tempo médio na fila.

#### Criar a fila

In [27]:
# Coloque seu código aqui
fila = FilaMG1(lb=25, E_S=0.034, DP_S=0.071)


#### Calcular o número médio de tarefas na fila

In [28]:
# Coloque seu código aqui (o resultado deve ser 12.910416666666674)

print (fila.E_Nq)

12.910416666666674


#### Calcular o tempo médio de espera na fila

In [29]:
# Coloque seu código aqui (o resultado deve ser 0.516416666666667)
print(fila.E_W)


0.516416666666667


## Rede de filas

## Medidas Operacionais
* $T$: Tempo total de observação
* $M$: Número de dispositivos no sistema
* $C$: Quantidade de chegadas ao sistema no tempo T
* $C_{i}$: Quantidade de chegadas ao subsistema i no tempo T
* $P$: Quantidade de partidas do sistema (solicitações tratadas) no tempo de observação
* $P_{i}$: Quantidade de partidas do subsistema i (solicitações tratadas) no tempo de observação
* $\lambda$: Taxa de chegada no sistema
* $\lambda_{i}$: Taxa de chegada no subsistema i                
* $\mu$: Vazão do sistema
* $V$: Vazão do subsistema i
* $D$: Demanda - tempo no sistema para atender uma tarefa (incluindo todas as visitas)
* $D_{i}$: Demanda - tempo no subsistema i para atender uma tarefa (incluindo todas as visitas)
* $V_{i}$: Número de visitas ao subsistema i necessárias para atender uma tarefa
* $N$: Número médio de tarefas no sistema
* $N_{i}$: Número médio de tarefas no subsistema i
* $R$: Tempo médio de resposta do sistema
* $R_{i}$: Tempo médio de resposta do subsistema i
* $S$: Tempo médio de serviço no sistema
* $S_{i}$: Tempo médio de serviço no subsistema i
* $B$: Tempo que o sistema permanece ocupado
* $B_{i}$: Tempo que o subsistema i permanece ocupado
* $U$: Taxa de utilização do sistema
* $U_{i}$: Taxa de utilização do subsistema i

## Relações Operacionais
* $\lambda_{i}=C_{i}/T$
* $\mu_{i}=P_{i}/T$
* $U_{i}=B_{i}/T$
* $S_{i}=B_{i}/P_{i}$
* $V_{i}=P_{i}/P$
* $\mu_{i}=\mu/V_{i}$


* $D=\sum_{}^{}D_{i}$
* $D_{i}=S_{i}\cdot V_{i}$
* $U_{i}=\mu_{i}\cdot S_{i}=\mu\cdot S_{i}\cdot V_{i}=\mu \cdot D_{i}$
* $N_{i}= U_{i} / (1 - U_{i})$
* $N_{i}= \mu_{i} \cdot R_{i}$
* $N=\sum_{}^{}N_{i}$
* $R_{i}= S_{i} / (1 - U_{i})$
* $R=\sum_{}^{} R_{i}\cdot V_{i}$

### Questão 5
Durante um período de observação de 60 segundos, 2.400.000 transações foram processadas em um servidor de autenticação. O tempo de processamento médio (tempo de serviço médio) é de 0,005 milissegundo. Calcular:
* Vazão do servidor ($\mu$)
* Taxa de utilização do servidor ($U$)
* Número médio de tarefas no servidor ($N$)
* Tempo de resposta médio ($R$)

Dados:
* $T = 60.000$ milissegundos
* $P = 2.400.000$ transações
* $S = 0,005$ milissegundos

In [ ]:
# Dados
T = 60000
P = 2400000
S = 0.005

In [ ]:
# Calcular e imprimir a vazão (MU)
# Coloque seu código aqui (o resultado deve ser 40.0)



In [ ]:
# Calcular e imprimir a taxa de utilização (U)
# Coloque seu código aqui (o resultado deve ser 0.2)



In [ ]:
# Calcular e imprimir o número médio de tarefas no servidor (N)
# Coloque seu código aqui (o resultado deve ser 0.25)



In [ ]:
# Calcular e imprimir o tempo de resposta médio (R)
# Coloque seu código aqui (o resultado deve ser 0.0062499999999999995)

